# 02 · Processing
## Levelling AFM data with masking

Raw AFM images are almost never flat. The scanner adds **tilt** and **bow**, and raster
scanning adds **line artifacts**. Before you can measure anything; heights, roughness,
volumes; you have to *level* the data so the background sits flat at zero.

If you fit the background to the *whole* image, tall features drag the fit upward and you
over-subtract around them. The fix is **masking**: detect the features, exclude them, and
fit from the remaining background pixels only.

We build the tools from scratch, then assemble them into a complete pipeline. Each tool is
demonstrated on the sample that needs it most.

| sample | artifact | tool |
|---|---|---|
| Annexin membrane protein | tilt | plane fit |
| Gallium droplets | line offsets | row-median alignment |
| DNA Origami (reloaded from nb 01) | tilt + line offsets | plane + masked line fit |
| DNA on mica | bow | plane + masked 2D polynomial |

**Contents**
1. Imports and data
2. The fitting functions
3. Masking the features
4. Building a complete pipeline
5. Choosing methods for *your* samples
6. Saving the result

---

## 1 · Imports and data

We are going to use NumPy and functions from scikit-learn for levelling as well as the otsu threshold algorithm from scikit-image. We are so going to load some more example data from the repo data folder and reload one of the images from notebook 01.

In [ ]:

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from skimage.filters import threshold_otsu

from AFMReader.general_loader import LoadFile

def register_afm_colormaps(folder=Path(".") / "resources" / "colormaps"):
    """Load the workshop's AFM colormaps from CSV and register them with matplotlib.
    Safe to call more than once — already-registered colormaps are skipped."""
    for name in ("afm_brown", "playnano_gold", "classic_afm", "aurea"):
        if name in plt.colormaps():
            continue
        rgb_data = np.loadtxt(folder / f"{name}.csv", delimiter=",")
        cmap = ListedColormap(rgb_data, name=name)
        plt.colormaps.register(cmap)
        plt.colormaps.register(cmap.reversed(), name=f"{name}_r")

register_afm_colormaps()

DATA_DIR  = Path(".") / "data" / "samples"
ARRAY_DIR = Path(".") / "data" / "arrays"
print("Ready.")


In [ ]:

# Sample 1: Annexin membrane
img_annexin, px_annexin = LoadFile(DATA_DIR / "save-2022.11.04-14.37.25.304.jpk",
                              channel="height_retrace").load()
img_annexin = img_annexin.astype(np.float64)

# Sample 2: Gallium droplets.
img_ga, px_ga = LoadFile(DATA_DIR / "5 min _45 nm hole.001.spm",
                            channel="Height Sensor").load()
img_ga = img_ga.astype(np.float64)

# Sample 3: DNA origami, reloaded from notebook 01
_d = np.load(ARRAY_DIR / "image.npz")
img_orig, px_orig = _d["image"].astype(np.float64), float(_d["pixel_to_nm"])

# Sample 4: DNA plasmids on mica
img_dna, px_dna = LoadFile(DATA_DIR / "20240513_rel_PJIP37_unknot_4ng_eph_ni__.0_00071.spm",
                            channel="Height").load()
img_dna = img_dna.astype(np.float64)

for name, im, px in [("Annexin", img_annexin, px_annexin),
                     ("Gallium droplet", img_ga, px_ga),
                     ("DNA origami",        img_orig,  px_orig),
                     ("DNA plasmids",      img_dna,   px_dna)]:
    print(f"{name:18s}: {im.shape}, {px:.3f} nm/pixel")
    

In [ ]:

def show(img, title, cmap="playnano_gold", zmax=None, zmin=None, ax=None, colorbar=True):
    ax = ax or plt.gca()
    im = ax.imshow(img, cmap=cmap, vmax=zmax, vmin=zmin, origin="lower")
    ax.set_title(title); ax.axis("off")
    if colorbar:
        plt.colorbar(im, ax=ax, label="Height (nm)", fraction=0.046)

fig, ax = plt.subplots(1, 4, figsize=(18, 4))
show(img_annexin,  "Annexin (tilt)",         ax=ax[0])
show(img_ga, f"Gallium droplets", ax=ax[1])
show(img_orig,  "DNA Origami",            ax=ax[2])
show(img_dna,   "DNA Plasmids",     ax=ax[3])
plt.tight_layout()
plt.show()


## 2 · The fitting functions

Each function fits a surface or per-row correction and returns it so we can subtract it from the data. 

We introduce each tool alongside the sample that needs it.

### 2.1 · Plane fit: removes tilt

AFM images often exhibit a background tilt because the sample and scanner are not perfectly perpendicular. To correct for this, we fit and subtract a plane from the image. Below, this is first shown step-by-step for the annexin sample, and then later wrapped into a reusable function.

---

There are several ways to fit a plane to a surface; here we use the `LinearRegression` class from `scikit-learn` to perform a least-squares fit to all pixel coordinates. We first convert the image into a set of data points: each pixel becomes a pair of coordinates `(x, y)` with an associated height value `z`. This is done by creating coordinate grids and flattening both the coordinates and the image into 1D arrays so that we have a list of `(x, y) -> z` samples.

We then fit a linear model of the form:

`z = a·x + b·y + c`

where `a` and `b` describe the tilt in the x and y directions, and `c` is a constant offset. The least-squares procedure finds the values of `a`, `b`, and `c` that minimise the total squared difference between the predicted plane and the actual height values at every pixel.

Once the model is fitted, we evaluate it at every `(x, y)` coordinate to reconstruct the best-fit plane as a 2D surface. Finally, we subtract this plane from the original image, removing the large-scale tilt and leaving only the local surface features of interest.

Before correction, the image shows a clear vertical gradient: the top appears lower (darker) than the bottom, indicating tilt. This is captured by the fitted plane (top right). Subtracting the plane produces a levelled image where the sample is aligned with the x–y plane. The improvement is also visible in the column profile, where the slope present in the raw data is removed after levelling.

In [ ]:

# Step by step on the tilt sample
h, w = img_annexin.shape
Y, X = np.mgrid[0:h, 0:w]                        # pixel coordinate grids
coords = np.column_stack([X.ravel(), Y.ravel()])  # shape (h*w, 2): one (x,y) per pixel
z = img_annexin.ravel()                              # heights in the same pixel order
model = LinearRegression().fit(coords, z)         # fit z = a*x + b*y + c

plane     = model.predict(coords).reshape(h, w)   # evaluate at every pixel -> 2D surface
flat_annexin = img_annexin - plane                      # subtract

# 2×2: original / fitted plane / levelled / column profile (tilt runs top-to-bottom here)
col = int(w // 5)
y_nm    = np.arange(h) * px_annexin
raw_col = img_annexin[:, col]

fig, ax = plt.subplots(2, 2, figsize=(10, 9))
show(img_annexin,  "1. Original (raw)",          ax=ax[0, 0])
show(plane,     "2. Fitted plane",            ax=ax[0, 1], colorbar=False)
show(flat_annexin, "3. Levelled (raw − plane)",  ax=ax[1, 0])
for a in (ax[0, 1], ax[1, 0]):
    a.axvline(col, color="r", lw=1, ls="--")
ax[1, 1].plot(y_nm, raw_col - raw_col.mean(), label="raw (mean-subtracted)", alpha=0.6)
ax[1, 1].plot(y_nm, flat_annexin[:, col],        label="plane-levelled")
ax[1, 1].axhline(0, color="k", lw=0.5)
ax[1, 1].legend()
ax[1, 1].set_xlabel("Distance (nm)"); ax[1, 1].set_ylabel("Height (nm)")
ax[1, 1].set_title("4. Column profile (tilt direction)")
ax[1, 1].set_box_aspect(1)
plt.tight_layout()
plt.show()


Now wrap it into a function. The `mask` argument is already wired in — we'll use
it in Section 4.

In [ ]:

def fit_plane(data, mask=None):
    """Return the best-fit plane. Fit background only if mask given (True=feature)."""
    h, w = data.shape
    Y, X = np.mgrid[0:h, 0:w]
    coords = np.column_stack([X.ravel(), Y.ravel()])
    z = data.ravel()
    keep = np.ones(z.shape, bool) if mask is None else ~mask.ravel()
    model = LinearRegression().fit(coords[keep], z[keep])
    return model.predict(coords).reshape(h, w)

assert np.allclose(img_annexin - fit_plane(img_annexin), flat_annexin)
print("fit_plane() ✓")


### 2.2 · Row-median alignment: removes line offsets

Each scan line can sit at a slightly different height due to feedback drift between lines.
Subtracting the **median** of each row removes that offset. For this we just use the standard NumPy median function.

The median is robust, a few tall feature pixels barely move it, whereas the mean would be dragged upward, leaving a dark shadow either side of each feature.

We demonstrate on an AFM image of nanoscale liquid gallium metal droplets, where the line noise is clearly visible as horizontal stripes. After a median row align the stripes are removed and the background is centred on 0.

In [ ]:

def row_median_align(data, mask=None):
    """Subtract each row's median (background pixels only, if mask given)."""
    out = data.copy()
    for i in range(data.shape[0]):   # loop over each line
        ref = data[i, :] if mask is None else data[i, ~mask[i, :]] 
        if ref.size == 0:
            ref = data[i, :]
        out[i, :] -= np.median(ref)   # subtract median of the line from the line
    return out

plane_fit = img_ga - fit_plane(img_ga)
ga_aligned = row_median_align(plane_fit, mask=None)

# Before/after + a row profile to show the offsets being removed
col_v = int(img_ga.shape[1] // 1.1)
x_nm  = np.arange(img_ga.shape[1]) * px_ga

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
show(plane_fit,    f"Gallium droplets image with plane fit (line offsets visible)", ax=ax[0], zmin=-5.0, zmax=5.0)
show(ga_aligned, "After row-median alignment",        ax=ax[1], zmin=-5.0, zmax=5.0)
for a in (ax[0], ax[1]):
    a.axvline(col_v, color="r", lw=1, ls="--")
ax[2].plot(x_nm, plane_fit[:, col_v],    label="raw",     alpha=0.6)
ax[2].plot(x_nm, ga_aligned[:, col_v], label="aligned")
ax[2].axhline(0, color="k", lw=0.5)
ax[2].legend(); ax[2].set_title(f"Row profile (column {col_v})")
ax[2].set_xlabel("Distance (nm)")
ax[2].set_ylabel("Height (nm)")
plt.tight_layout()
plt.show()


### 2.3 · Per-row polynomial fit: removes line tilt and bow

`row_median_align` removes a single *constant* per row. A **polynomial line fit** goes further: it removes per-row *slope* (order 1) or *curvature* (order 2) as well, which is useful when each scan line has its own residual tilt or wobble after the plane is removed.

We use the NumPy `polyfit` function to calculate the polynomial coefficients for the unmasked pixels in the line and then `polyval` to evaluate the polynomial for all pixels.

In [ ]:

def row_polynomial_align(data, order=1, mask=None):
    """Fit and subtract a polynomial along each row (background only if masked).

    order=1 removes per-row tilt; order=2 also removes per-row bow.
    Falls back to median for rows with too few background pixels to fit.
    """
    out = data.astype(np.float64).copy()
    x   = np.arange(data.shape[1])
    for i in range(data.shape[0]):
        keep = np.ones(x.shape, bool) if mask is None else ~mask[i, :]
        if keep.sum() <= order:
            out[i, :] -= np.median(data[i, keep] if keep.any() else data[i, :])
            continue
        coeffs = np.polyfit(x[keep], data[i, keep], order)   # Calculate polynomial coefficients using least squares (np.polyfit())
        out[i, :] -= np.polyval(coeffs, x)   # evaluate the polynomial at each x values and subtract it from the line
    return out
print("row_polynomial_align() ✓")


### 2.4 · 2D polynomial fit: removes bow

When the whole image has a gentle *curve* (scanner bow) a plane can't follow it, but a
2D polynomial can. We build the polynomial terms with `PolynomialFeatures` and solve with
`np.linalg.lstsq`.

In [ ]:

def fit_polynomial(data, order=2, mask=None):
    """Return best-fit 2D polynomial surface (background only if masked)."""

    h, w = data.shape   # Get image dimensions
    Y, X = np.mgrid[0:h, 0:w]   # Create coordinate grids
    coords = np.column_stack([X.ravel(), Y.ravel()])   # Flatten coordinates into a list of (x, y) pairs
    z = data.ravel()    # Flatten image into a 1D array of height values
    A = PolynomialFeatures(order).fit_transform(coords)   # Each (x, y) becomes polynomial features like: [1, x, y, x², x*y, y², ...] depending on 'order'
    keep = np.ones(z.shape, bool) if mask is None else ~mask.ravel()
    coeff, *_ = np.linalg.lstsq(A[keep], z[keep], rcond=None)   # solve least-squares
    fitted = A @ coeff   # Multiply the feature matrix by the coefficient vector.
    return fitted.reshape(h, w)   # Reshape back to 2D image form

print("fit_polynomial() ✓")


## 3 · Masking the features

Sometimes high features in an image can distort the fitting and create artifacts. To avoid this the features can be masked so that only the background (that should be flat) is used to calculate the fits. 

All four functions above accept an optional **`mask`** parameter that is a binary image with the features marked as `True` (and excluded from fits) and the background masked as `False` and used in fitting. 

Here's how to make masks:

### 3.1 · Threshold mask (mean + k·std)

Flag any pixel more than `k` standard deviations above the mean. Demonstrated on the DNA origami sample, rough-levelled with the plane fit and median row align so the threshold sees features, not tilt.

In [ ]:

def threshold_mask(data, k=1.0):
    """Boolean mask: True where height > mean + k*std (True = feature)."""
    return data > (data.mean() + k * data.std())

# Rough-level first (plane + median rows) so the threshold is meaningful
rough_orig = row_median_align(img_orig - fit_plane(img_orig))
mask0 = threshold_mask(rough_orig, k=1.0)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
show(rough_orig, "Origami, rough-levelled", ax=ax[0], zmin=-2.0, zmax=3.0)
ax[1].imshow(mask0, cmap="gray", origin="lower")
ax[1].set_title(f"Threshold mask  k=1.0  (covers {100*mask0.mean():.1f}%)")
ax[1].axis("off")
plt.tight_layout()
plt.show()


### 3.2 · Otsu: automatic threshold

Otsu's method picks the threshold that best separates background from features by minimising within-class variance; one call from `scikit-image`, no `k` to tune.

In [ ]:

otsu_t    = threshold_otsu(rough_orig)
mask_otsu = rough_orig > otsu_t

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].hist(rough_orig.ravel(), bins=120)
ax[0].axvline(otsu_t, color="r")
ax[0].set_title(f"Histogram  (Otsu = {otsu_t:.2f} nm)")
ax[0].set_xlabel("Height (nm)")
ax[1].imshow(mask0,     cmap="gray", origin="lower")
ax[1].set_title("mean+std  (k=1.0)"); ax[1].axis("off")
ax[2].imshow(mask_otsu, cmap="gray", origin="lower")
ax[2].set_title("Otsu"); ax[2].axis("off")
plt.tight_layout()
plt.show()


### Exercise

Try `k = 0.5`, `1.0`, `2.0` in `threshold_mask` on `rough_orig`. Too low grabs
background noise; too high misses faint features. Which do you trust most? Compare to
the Otsu mask.

Next, load and see the best threshold for your own data.

In [ ]:

# Your turn:
# for k in (0.5, 1.0, 2.0):
#     m = threshold_mask(rough_orig, k=k)
#     print(f"k={k}: mask covers {100*m.mean():.1f}%")
    
# thresh_o = threshold_otsu(rough_orig)
# mask_orig_otsu = rough_orig > thresh_o
# print(f"Otsu: mask covers {100*mask_orig_otsu.mean():.1f}%")


## 4 · Building a complete pipeline


### The chicken-and-egg problem

For accurate levelling, we want to threshold by height to find features; but on **raw tilted data** the tilt itself can be taller than the threshold on one side of the image, so a naïve threshold grabs a wedge of background, not the features. The fix: **rough-level first**, threshold *that*, then use the mask to re-level properly.

The key idea: **rough-level to find features → mask → re-level from background only.**

We build two pipelines, each matched to its sample's main artifact.

### 4.1 · Plane + masked row polynomial → origami sample

The origami image has **tilt** and **line offsets**. A plane removes the global tilt; a masked per-row polynomial (order 1) then removes residual per-line tilt without distorting the features.

Below we compare this levelling routine with masking with a naive, rough level, with just a plane fit and row align.

In [ ]:

# STEP 1: rough level (plane + median rows)
rough = row_median_align(img_orig - fit_plane(img_orig))
# STEP 2: mask features from the rough-levelled image
mask_orig = threshold_mask(rough, k=1.0)
# STEP 3: re-fit plane to background only
levelled_orig = img_orig - fit_plane(img_orig, mask=mask_orig)
# STEP 4: re-align rows to background only with order-1 polynomial
levelled_orig = row_polynomial_align(levelled_orig, order=1, mask=mask_orig)

# Compare naive vs pipeline
naive_orig = row_median_align(img_orig - fit_plane(img_orig))
bg = ~mask_orig
print(f"Background mean - naive:    {naive_orig[bg].mean():+.3f} nm")
print(f"Background mean - pipeline: {levelled_orig[bg].mean():+.3f} nm")

row = int(img_orig.shape[0] // 2.4)
x_nm = np.arange(img_orig.shape[1]) * px_orig
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
show(naive_orig,    "Naive (no mask)", ax=ax[0], zmin=-2.0, zmax=3.0)
show(levelled_orig, "Pipeline",        ax=ax[1], zmin=-2.0, zmax=3.0)
for a in (ax[0], ax[1]):
    a.axhline(row, color="r", lw=1, ls="--")
ax[2].plot(x_nm, naive_orig[row, :],    label="naive",    alpha=0.6)
ax[2].plot(x_nm, levelled_orig[row, :], label="pipeline")
ax[2].axhline(0, color="k", lw=0.5)
ax[2].legend(); ax[2].set_title(f"Row profile (row {row})")
ax[2].set_xlabel("Distance (nm)"); ax[2].set_ylabel("Height (nm)")
plt.tight_layout()
plt.show()


### 4.2 · Plane + masked 2D polynomial → DNA plasmids (bow)

The DNA plasmid image has **bow**; gentle curvature across the whole frame that a plane can't follow. A masked 2D polynomial handles this, but the mask is essential: without it the polynomial bends to follow the features instead of the substrate.

In [ ]:

# STEP 1: rough level (plane + median rows)
rough_dna = row_median_align(img_dna - fit_plane(img_dna))
# STEP 2: mask features (higher k here: DNA is small so we cast a tight net)
k_dna = 2.0
mask_dna = threshold_mask(rough_dna, k=k_dna)
# STEP 3: 2D polynomial fit to background only
plane_surface_dna = fit_plane(img_dna, mask=mask_dna)
poly_surface_dna  = fit_polynomial(img_dna, order=2, mask=mask_dna)
levelled_dna = img_dna - poly_surface_dna
# STEP 4: masked row alignment to clean up residual line offsets
levelled_dna = row_median_align(levelled_dna, mask=mask_dna)

# Compare masked plane vs masked polynomial to make the bow correction visible
flat_plane_dna = row_median_align(img_dna - plane_surface_dna, mask=mask_dna)

row_d = int(img_dna.shape[0] // 7)
x_nm  = np.arange(img_dna.shape[1]) * px_dna

fig, ax = plt.subplots(2, 2, figsize=(9, 10))
show(rough_dna,    "DNA plasmids: plane fit and median row align", ax=ax[0, 0], cmap="afm_brown", zmin=-2.0, zmax=3.0)
show(levelled_dna, "DNA plasmids: masked polynomial",              ax=ax[1, 0], cmap="afm_brown", zmin=-2.0, zmax=3.0)
ax[0, 0].axhline(row_d, color="r", lw=1, ls="--")
ax[1, 0].axhline(row_d, color="r", lw=1, ls="--")

ax[0, 1].imshow(mask_dna, cmap="gray", origin="lower")
ax[0, 1].set_title(f"Mask  k={k_dna}  (covers {100*mask_dna.mean():.1f}%)")
ax[0, 1].axis("off")

# Same two lines as before, with the captured bow curve drawn on top
bow_curve = (poly_surface_dna - plane_surface_dna)[row_d, :]   # the curvature the polynomial added beyond the plane
bow_curve -= bow_curve.mean()                                    # centre it for visual overlay only

ax[1, 1].plot(x_nm, flat_plane_dna[row_d, :], label="masked plane (bow remains)", lw=0.5)
ax[1, 1].plot(x_nm, levelled_dna[row_d, :],   label="masked polynomial (bow removed)", lw=0.5)
ax[1, 1].plot(x_nm, bow_curve, label="bow curve fitted (poly − plane)", lw=1.5, ls="--", color="green")
ax[1, 1].axhline(0, color="k", lw=0.5)
ax[1, 1].legend()
ax[1, 1].set_title("Plane vs polynomial: background")
ax[1, 1].set_xlabel("Distance (nm)")
ax[1, 1].set_ylabel("Height (nm)")
ax[1, 1].set_box_aspect(1)
plt.tight_layout()
plt.show()


### 4.3 · Wrap it up

Here are the two pipelines as clean, reusable functions.

In [ ]:

def level_plane_lines(img, k=1.0, line_order=1):
    """Plane + masked per-row polynomial. Best for sparse features + line offsets."""
    rough = row_median_align(img - fit_plane(img))
    mask  = threshold_mask(rough, k=k)
    out   = img - fit_plane(img, mask=mask)
    out   = row_polynomial_align(out, order=line_order, mask=mask)
    return out, mask

def level_polynomial(img, k=1.0, order=2):
    """Plane + masked 2D polynomial + masked row alignment. Best for bow."""
    rough = row_median_align(img - fit_plane(img))
    mask  = threshold_mask(rough, k=k)
    out   = img - fit_polynomial(img, order=order, mask=mask)
    out   = row_median_align(out, mask=mask)
    return out, mask

# Verify both reproduce the step-by-step results above
lo, mo = level_plane_lines(img_orig, k=1.0,   line_order=1)
ld, md_ = level_polynomial(img_dna,  k=k_dna, order=2)
assert np.allclose(lo, levelled_orig)
assert np.allclose(ld, levelled_dna)
print("Both pipeline functions verified ✓")


## 5 · Choosing methods for *your* sample

| sample type | main artifact | recommended pipeline |
|---|---|---|
| Sparse biomolecules on mica | tilt + line offsets | `level_plane_lines` |
| Same, with visible bow | bow | `level_polynomial` |
| Rough / granular (perovskite, grains) | no clear background | polynomial; mask only tallest grains |
| High-coverage film | film covers most of frame | polynomial; use a separate bare-substrate scan if possible |

Rule of thumb: **mask when features are sparse and tall; trust the background fit less as
coverage rises. Always sanity-check with a line profile.**

### Exercise 

Now try opening one of your own images (or another from the repo sample folder) and use these tools to level it.

In [ ]:

# Load data
# your_img, your_px = LoadFile(r"PATH/TO/YOUR/DATA.jpk",
#                               channel="height_retrace").load()
# your_img = your_img.astype(np.float64)

# Level data
# your_leveled, your_mask = level_plane_lines(your_img, k=1.0,   line_order=1)
# or
# your_leveled, your_mask = level_polynomial(your_img, k=1.0, order=2)
# or build your own pipeline from the functions above or others you know.

# Plot
# fig, ax = plt.subplots(1, 3, figsize=(14,6))
# show(your_img, "RAW", ax=ax[0])
# show(your_mask, "MASK", ax=ax[1])
# show(your_leveled, "LEVEL", ax=ax[2])



## 6 · Saving the levelled result

Save the origami and DNA results for the analysis notebook.

In [ ]:

ARRAY_DIR.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    ARRAY_DIR / "levelled_annexin.npz",
    image=flat_annexin, mask=None, pixel_to_nm=px_annexin,
)
np.savez_compressed(
    ARRAY_DIR / "levelled_origami.npz",
    image=levelled_orig, mask=mask_orig, pixel_to_nm=px_orig,
)
np.savez_compressed(
    ARRAY_DIR / "levelled_dna.npz",
    image=levelled_dna, mask=mask_dna, pixel_to_nm=px_dna,
)
print("Saved levelled_origami.npz and levelled_dna.npz")

for fname in ("levelled_origami.npz", "levelled_dna.npz", "levelled_annexin.npz"):
    d = np.load(ARRAY_DIR / fname)
    print(f"{fname}: keys = {list(d.keys())}")
    

In [ ]:

# Save your own levelled data

# np.savez_compressed(
#     ARRAY_DIR / "levelled_your_data.npz",
#     image=your_leveled, mask=your_mask, pixel_to_nm=your_px,
# )
# print("Saved levelled_your_data.npz")
# y_d = np.load(ARRAY_DIR / "levelled_your_data.npz")
# print(f"levelled_your_data.npz: keys = {list(y_d.keys())}")


---

### Recap

**Tools:** `fit_plane` · `row_median_align` · `row_polynomial_align` · `fit_polynomial`
— all take the same `mask` argument; `True` = feature (excluded), `False` = background.

**Workflow:** rough plane + row level → threshold mask → re-fit to background only.

**Match method to sample:**
- Tilt + line offsets → `level_plane_lines` (plane + masked `row_polynomial_align`)
- Bow → `level_polynomial` (masked 2D polynomial + masked row alignment)

**Next:** `03_analysis.ipynb` measures the levelled data: heights, profiles, roughness.